In [15]:
def create_custom_array(N_double):
    if N_double< 2:
        raise ValueError("N 必须 >= 2")
    
    arr = np.zeros(N_double)  # 初始化全0数组
    arr[0] = y1*1.5         # 首元素 = 4
    arr[-1] = y1*1.5        # 尾元素 = 5
    return arr

# 示例
N_double = 10
result = create_custom_array(N_double)
print(result)

[3. 0. 0. 0. 0. 0. 0. 0. 0. 3.]


In [16]:
import numpy as np

h1 = 1
m = 0.5

def coupling(matrix_1, matrix_2):
    s11_1, s12_1 = matrix_1[0, 0], matrix_1[0, 1]
    s21_1, s22_1 = matrix_1[1, 0], matrix_1[1, 1]
    s11_2, s12_2 = matrix_2[0, 0], matrix_2[0, 1]
    s21_2, s22_2 = matrix_2[1, 0], matrix_2[1, 1]
    
    denom = 1 - s11_2 * s22_1
    s11 = s11_1 + s12_1 * s21_1 * s11_2 / denom
    s12 = s12_1 * s12_2 / denom
    s21 = s21_2 * s21_1 / denom
    s22 = s22_2 + s21_2 * s12_2 * s22_1 / denom
    
    return np.array([[s11, s12], [s21, s22]], dtype=np.complex128)

def multiple_coupling(matrix_collect):
    result = matrix_collect[0]
    for matrix in matrix_collect[1:]:
        result = coupling(result, matrix)
    return result

def coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a):
    constant_1 = 2 * m * (y1 + 1j * y2) / (h1**2)
    constant_2 = 2 * m * (y1 - 1j * y2) / (h1**2)
    
    di = np.array([constant_1 if i % 2 == 0 or i >= 2 * N and i < 2 * N + n3 else constant_2 
                   for i in range(2 * N + n3 + n4)], dtype=complex)

    N_double=2*N+n3+n4
    di=di+create_custom_array(N_double)
    
    x_0 = 0
    x_1 = a * (2 * N + n3 + n4) - a + x_0
    location = np.linspace(x_0, x_1, 2 * N + n3 + n4)
 #   print(location)
    k = np.sqrt(E * 2 * m) / h1
    exp_2jk = np.exp(2j * k * location)
    exp_minus_2jk = np.exp(-2j * k * location)
    
    matrix_collect = np.zeros((2 * N + n3 + n4, 2, 2), dtype=complex)
    for i in range(2 * N + n3 + n4):
        s11 = di[i] * exp_2jk[i]
        s12 = 2j * k
        s21 = 2j * k
        s22 = di[i] * exp_minus_2jk[i]
        
        matrix_collect[i] = np.array([[s11, s12], [s21, s22]]) / (-di[i] + 2j * k)
  #  print(matrix_collect)
    return multiple_coupling(matrix_collect)

def Transmission_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[1, 0])**2)

def Reflection_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a):
    matrix = coupling_matrix_of_N_modified(E, y1, y2, N, n3, n4, a)
    return float(np.abs(matrix[0, 0])**2)

In [17]:
import numpy as np                         #T&R          #最终25.4.10
import matplotlib.pyplot as plt
import os
# 绘图参数        
N = 100
n3 = 0
n4 = 0
y11 = [2]  # y1 的不同取值
y22 = [2]     # y2 的不同取值
a = 0.5
zhi_1 = ['T']  # 选择绘制 T（透射系数）

# 能量范围
E_ranges = [(40,170)]  # 可以分段计算

# 输出文件夹
folder_name = "T&R实数与虚数_3"
output_dir = os.path.join(r"C:\Users\taoji\Desktop\结果\2025.4.12后_Python运行结果", folder_name)

# 确保输出目录存在
try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"文件夹 '{output_dir}' 已创建或已存在")
except Exception as e:
    print(f"创建文件夹时出错: {e}")

# 遍历每个能量范围
for range_idx, (E_min, E_max) in enumerate(E_ranges):
    # 动态子图布局
    num_rows = len(y11)
    num_cols = len(y22)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(8 * num_cols, 4 * num_rows))
    fig.suptitle(f'Transmission and Reflection Coefficients vs Energy ({E_min} to {E_max})', fontsize=16)

    # 处理单行/单列情况
    if num_rows == 1 and num_cols == 1:
        axes = np.array([[axes]])
    elif num_rows == 1:
        axes = axes.reshape(1, -1)
    elif num_cols == 1:
        axes = axes.reshape(-1, 1)

    # 遍历 y11 和 y22 的组合
    for i, y1 in enumerate(y11):
        for j, y2 in enumerate(y22):
            E_values = np.linspace(E_min, E_max, 1000)
            T_values = [Transmission_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a) for E in E_values]
            R_values = [Reflection_coefficient_of_N_modified(E, y1, y2, N, n3, n4, a) for E in E_values]
            Sum_values = [T + R for T, R in zip(T_values, R_values)]

            #临时存放
            #tao=dict(zip(E_values,T_values))

            
            ax = axes[i, j]
            if 'T' in zhi_1:
                ax.plot(E_values, T_values, label='Transmission (T)', color='blue')
            if 'R' in zhi_1:
                ax.plot(E_values, R_values, label='Reflection (R)', color='red')
            if 'T+R' in zhi_1:
                ax.plot(E_values, Sum_values, label='T + R', color='gray', linestyle='-')

            ax.set_xlabel('Energy (E)')
            ax.set_ylabel('Coefficient Value')
            ax.set_title(f'y1={y1}, y2={y2}')

            
            #ax.axhline(y=1, color='green', linestyle='-', linewidth=1, alpha=0.7)
            #ax.axvline(x=39.478417604357, color='green', linestyle='-', linewidth=1, alpha=0.7)
            #ax.axvline(x=47.240296550706, color='green', linestyle='-', linewidth=1, alpha=0.7)
            #ax.axvline(x=157.913670417430, color='green', linestyle='-', linewidth=1, alpha=0.7)
            #ax.axvline(x=165.848348494293, color='green', linestyle='-', linewidth=1, alpha=0.7)
            
            ax.legend()
            ax.grid(True)
            ax.set_xlim(E_min, E_max)
            ax.set_ylim(0, 2)

    plt.tight_layout()

    # 保存图片
    y11_str = '_'.join(map(str, y11))
    y22_str = '_'.join(map(str, y22))
    zhi1_str = '_'.join(zhi_1)
    output_filename = os.path.join(
        output_dir,
        f'{range_idx + 1}_T&R({E_min},{E_max})_N={N}_n3={n3}_n4={n4}_y11={y11_str}_y22={y22_str}_a={a}_zhi1={zhi1_str}.png'
    )
    plt.savefig(output_filename, dpi=300)
    plt.close()
    print(f"已保存: {output_filename}")

文件夹 'C:\Users\taoji\Desktop\结果\2025.4.12后_Python运行结果\T&R实数与虚数_3' 已创建或已存在
已保存: C:\Users\taoji\Desktop\结果\2025.4.12后_Python运行结果\T&R实数与虚数_3\1_T&R(40,170)_N=100_n3=0_n4=0_y11=2_y22=2_a=0.5_zhi1=T.png
